In [ ]:
import requests
import json
import pandas as pd
import time

In [ ]:
#api_url = 'https://footballapi.pulselive.com/football/competitions/1/compseasons?page=0&pageSize=100'
# 24/25 Season
#           'https://footballapi.pulselive.com/football/standings?compSeasons=719&altIds=true&detail=2&FOOTBALL_COMPETITION=1&live=true'
# 23/24 Season
#api_url2 = 'https://footballapi.pulselive.com/football/standings?compSeasons=578&altIds=true&detail=2&FOOTBALL_COMPETITION=1'

In [ ]:
def req_and_parse_data(url, headers, years):
  # GET request to API
  resp = requests.get(url, headers=headers)

  # Parse JSON response and pretty print
  if resp.status_code == 200:
    data = resp.json()
    #print(json.dumps(data, indent=4, sort_keys=True))

    # Get table data
    table_data = data['tables'][0]['entries']

    position = []
    club = []
    abb_club = []
    points = []
    played = []
    won = []
    lost = []
    drawn = []
    gd = []
    gf = []
    ga = []
    season = []

    # Populate lists
    for entry in table_data:
        position.append(entry['position'])
        club.append(entry['team']['name'])
        abb_club.append(entry['team']['club']['abbr'])
        points.append(entry['overall']['points'])
        played.append(entry['overall']['played'])
        won.append(entry['overall']['won'])
        lost.append(entry['overall']['lost'])
        drawn.append(entry['overall']['drawn'])
        gd.append(entry['overall']['goalsDifference'])
        gf.append(entry['overall']['goalsFor'])
        ga.append(entry['overall']['goalsAgainst'])
        season.append(years)

    # Create a dictionary for dataframe
    season_data = {'Position': position, 'Team': club, 'Abb': abb_club, 'Points': points, 'Played': played, 'Won': won, 'Lost': lost, 'Draw': drawn, 'GD': gd, 'GF': gf, 'GA': ga, 'Season': season}

    table = pd.DataFrame(season_data)
    # Set position column as dataframe index
    table.set_index('Position', inplace=True)

    return table

  else:
    print(f'Status code: {resp.status_code}')
    return None

In [ ]:
#df = req_and_parse_data(api_url2 ,headers)

#df.head()

In [ ]:
def seasons_url(seasons, years):
  main_url = 'https://footballapi.pulselive.com/football/standings?altIds=true&detail=1&FOOTBALL_COMPETITION=1&comps=1'
  n = 53
  all_seasons = []

  # Loop through each season and relevant url
  for i in range(len(seasons)):
    season = seasons[i]
    year = years[i]

    # Modify url for each season
    api_url = main_url[:n] + 'compSeasons=' + str(season) + '&' + main_url[n:]
    api_url = api_url.replace( 'detail=1', 'detail=2' )
    api_url = api_url.removesuffix('&comps=1')

    df = req_and_parse_data(api_url, headers, year)
    if df is not None:
      all_seasons.append(df)
    time.sleep(2)

  return pd.concat( all_seasons, ignore_index=True )


In [ ]:
# Headers to mimic browser request to get an API response
headers = {
    'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    'Accept-Language': 'en-GB,en-US;q=0.9,en;q=0.8',
    'Referer': 'https://www.premierleague.com/',
    'Origin': 'https://www.premierleague.com'
}

In [ ]:
seasons = [777, 719, 578, 489, 418, 363, 274]
years = ['25/26','24/25', '23/24', '22/23', '21/22', '20/21', '19/20']

In [ ]:
df = seasons_url(seasons, years)
#print(df)
df.to_csv('20_26_PLTable.csv', index=False)